In [1]:
import numpy as np
import pandas as pd

RUTA = "../datasets"
# DATASET = "LI-Small_Trans.csv"
DATASET = "transacciones_sample_q4.csv"
ACCOUNT = "LI-Small_accounts.csv"

trans_df = pd.read_csv(f"{RUTA}/{DATASET}")
print("SIZE:", trans_df.size)
accounts_df = pd.read_csv(f"{RUTA}/{ACCOUNT}")
print("SIZE:", accounts_df.size)
trans_df.head(5)

SIZE: 76164671
SIZE: 3563440


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:08,11,8000ECA90,11,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
1,2022/09/01 00:21,3402,80021DAD0,3402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,0
2,2022/09/01 00:00,11,8000ECA90,1120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0
3,2022/09/01 00:16,3814,8006AD080,3814,8006AD080,12.32,US Dollar,12.32,US Dollar,Reinvestment,0
4,2022/09/01 00:00,20,8006AD530,20,8006AD530,2941.56,US Dollar,2941.56,US Dollar,Reinvestment,0


In [2]:
# Analyze timestamps.
print(f"Timestamp range: [{trans_df["Timestamp"].min()},{trans_df["Timestamp"].max()}]")

Timestamp range: [2022/09/01 00:00,2022/09/17 15:28]


In [3]:
# Analyze transfers. Check for duplicate Account Numbers in different banks.
df_senders = trans_df[['From Bank', 'Account']].rename(columns={
    'From Bank': 'Bank', 
})
df_receivers = trans_df[['To Bank', 'Account.1']].rename(columns={
    'To Bank': 'Bank', 
    'Account.1': 'Account'
})
df_bank_accounts = pd.concat([df_senders, df_receivers],ignore_index=True)
df_bank_counts = df_bank_accounts.drop_duplicates().groupby('Account')['Bank'].count()
df_bank_counts[df_bank_counts > 1]

Account
817037B20    2
817038DF0    2
8177C8ED0    2
8177C94B0    2
Name: Bank, dtype: int64

In [4]:
#Filter non USD transactions.
trans_usd_df = trans_df[trans_df['Payment Currency'] == "US Dollar"]
print("SIZE:", trans_usd_df.shape[0])

SIZE: 2553899


In [5]:
trans_usd_sept_1st_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/01') & (trans_usd_df["Timestamp"] <= '2022/09/06')]
print("SIZE:", trans_usd_sept_1st_df.shape[0])

SIZE: 1393082


In [6]:
ranged_trans_usd_sept_df = trans_usd_sept_1st_df\
    .groupby(["From Bank", "Account"])\
    .filter(lambda x: x.groupby(["To Bank", "Account.1"]).size().size > 5)
print("SIZE:", ranged_trans_usd_sept_df.shape[0])

SIZE: 406050


In [7]:
#1. Amount, source and target accounts for transactions of less than 50 USD.

low_profile_transactions = trans_usd_df[trans_usd_df['Amount Paid'] < 50]
low_profile_transactions = low_profile_transactions[['From Bank', 'Account', 'To Bank','Account.1', 'Amount Paid']]
low_profile_transactions.to_csv("q1_solucion.csv", index=False)
low_profile_transactions

,From Bank,Account,To Bank,Account.1,Amount Paid
3,3814,8006AD080,3814,8006AD080,12.32
7,11,8000ECA90,11,8000ECA90,22.97
8,1120,8006AA910,243166,81470DCF0,43.53
9,1217,8006AD4E0,1217,8006AD4E0,5.04
10,224,8006AD580,23319,80567ED00,9.28
...,...,...,...,...,...
6923281,224,80011C480,224,80011C480,28.50
6923886,236488,814B886B1,236488,814B886B0,0.39
6924019,13474,803A93631,13474,803A93630,0.08
6924021,13474,803A93631,13474,803A93630,0.23


In [8]:
#2. Max amount by source bank, source Bank Id and Bank Name considering all the transactions.

max_amount_trans_usd_idx = trans_usd_df.groupby(["From Bank"])["Amount Paid"].idxmax()
max_amount_trans_usd = trans_usd_df.loc[max_amount_trans_usd_idx]
max_amount_bank = max_amount_trans_usd.merge(accounts_df, left_on="From Bank", right_on="Bank ID")
q2_solucion = max_amount_bank[["From Bank", "Account", "Bank Name","Amount Paid"]].drop_duplicates()
q2_solucion.to_csv("q2_solucion.csv", index=False)
# q2_solucion

In [9]:
#3. Source account, payment format, and amount of transactions in period [2022-09-06, 2022-11-06] with amount lower than AVG/100 of period [2022-09-01, 2022-09-05] for the same type of transaction.

avg_amounts_per_type = trans_usd_sept_1st_df.groupby(["Payment Format"])["Amount Paid"].mean().reset_index()
trans_usd_sept_2nd_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/06') & (trans_usd_df["Timestamp"] <= '2022/09/15')]
trans_usd_sept_2nd_with_avg_df = trans_usd_sept_2nd_df.merge(avg_amounts_per_type, left_on=["Payment Format"], right_on=["Payment Format"]).rename(columns={
    "Amount Paid_x": "Amount Paid",
    "Amount Paid_y": "AVG",
})
lower_trans_usd_sept_2nd_with_avg_df = trans_usd_sept_2nd_with_avg_df[trans_usd_sept_2nd_with_avg_df["Amount Paid"] < trans_usd_sept_2nd_with_avg_df["AVG"] * 0.01]
q3_solucion = lower_trans_usd_sept_2nd_with_avg_df[["From Bank", "Account", "Payment Format", "Amount Paid"]]
q3_solucion.to_csv("q3_solucion.csv", index=False)
# q3_solucion

In [10]:
#4. Accounts that match the scatter-gather pattern and where the source account has transferred to more than 5 distinct accounts.

# Aristas A→B: solo desde cuentas scatter-calificadas (A envió a > 5 destinos distintos)
accounts_ab = ranged_trans_usd_sept_df[["From Bank", "Account", "To Bank", "Account.1"]]
# Aristas B→C: TODAS las transacciones USD del período.
# Bug original: usar ranged_trans_usd_sept_df para ambos lados del merge excluía
# los intermediarios B que solo enviaban a 1 destino, haciendo que nunca se
# encontraran patrones scatter-gather en la práctica.
accounts_bc = trans_usd_sept_1st_df[["From Bank", "Account", "To Bank", "Account.1"]]

account_pairs_df = accounts_ab.merge(accounts_bc, left_on=["To Bank", "Account.1"], right_on=["From Bank", "Account"]).rename(columns={
    "From Bank_x": "From Bank",
    "Account_x": "From Account",
    "To Bank_y": "To Bank",
    "Account.1_y": "To Account"
})
account_pairs_df = account_pairs_df[(account_pairs_df["From Bank"] != account_pairs_df["To Bank"]) | (account_pairs_df["From Account"] != account_pairs_df["To Account"])]
# Deduplicar triplets (A, B, C) para no contar el mismo intermediario más de una vez
account_pairs_df = account_pairs_df.drop_duplicates(subset=["From Bank", "From Account", "To Bank_x", "Account.1_x", "To Bank", "To Account"])
account_pairs_df = account_pairs_df.groupby(["From Bank", "From Account", "To Bank", "To Account"], as_index=False).size()
account_pairs_df = account_pairs_df[(account_pairs_df["size"] > 5)]


from_account_pairs_df = account_pairs_df[["From Bank", "From Account"]].rename(columns={
    "From Bank": "Bank",
    "From Account": "Account"
})
to_account_pairs_df = account_pairs_df[["To Bank", "To Account"]].rename(columns={
    "To Bank": "Bank",
    "To Account": "Account"
})
unique_accounts = pd.concat([from_account_pairs_df, to_account_pairs_df]).drop_duplicates()
unique_accounts.to_csv("q4_solucion.csv", index=False)
unique_accounts

,Bank,Account
132408,1394,800701D70
226641,990001,SG_SRC_AA38607A
132408,11495,800A08E20
226641,990099,SG_DST_AA38607A


In [11]:
#Conversion dates of period [2022-09-01, 2022-09-05] with base USD
#Bitcoin rates taken from investing.com
#Rest of currencies from api.frankfurter.dev
conversion_rates_records = np.rec.array([
           ('2022/09/01', 1.4644, 5.1805, 1.314 , 0.97999, 6.9   , 1.0002, 0.86272, 3.3535, 79.543, 139.34, 20.189, 60.367, 3.75, 1.,  19793.1),
           ('2022/09/02', 1.4691, 5.2035, 1.3141, 0.98175, 6.9035, 1.0011, 0.86468, 3.3755, 79.719, 140.11, 20.085, 60.427, 3.75, 1., 199999. ),
           ('2022/09/03', 1.4691, 5.2056, 1.3138, 0.98207, 6.9046, 1.0013, 0.86478, 3.3791, 79.75 , 140.17, 20.081, 60.471, 3.75, 1.,  19831.4),
           ('2022/09/04', 1.4695, 5.2082, 1.3139, 0.98219, 6.9047, 1.0013, 0.8649 , 3.3815, 79.754, 140.22, 20.084, 60.461, 3.75, 1.,  19952.7),
           ('2022/09/05', 1.4722, 5.1786, 1.3142, 0.98273, 6.9216, 1.0068, 0.86813, 3.4006, 79.816, 140.49, 20.018, 60.737, 3.75, 1.,  20126.1)],
          dtype=[ ('Date', 'O'), ('Australian Dollar', '<f8'), ('Brazil Real', '<f8'), ('Canadian Dollar', '<f8'), ('Swiss Franc', '<f8'), ('Yuan', '<f8'), ('Euro', '<f8'), ('UK Pound', '<f8'), ('Shekel', '<f8'), ('Rupee', '<f8'), ('Yen', '<f8'), ('Mexican Peso', '<f8'), ('Ruble', '<f8'), ('Saudi Riyal', '<f8'), ('US Dollar', '<f8'), ('Bitcoin', '<f8')])
conversion_rates_df = pd.DataFrame.from_records(conversion_rates_records)
conversion_rates_df = conversion_rates_df.set_index("Date")

In [12]:
#5. Count of transactions of period [2022-09-01, 2022-09-05] with type Wire or ACH, having converted amount for that day less than USD 1.
trans_sept_1st_df = trans_df[(trans_df["Timestamp"] >= '2022/09/01') & (trans_df["Timestamp"] <= '2022/09/06')]
trans_sept_1st_wire_or_ach_df = trans_sept_1st_df[(trans_sept_1st_df["Payment Format"] == "Wire") | (trans_sept_1st_df["Payment Format"] == "ACH")]
trans_sept_1st_wire_or_ach_converted_df = trans_sept_1st_wire_or_ach_df.copy()
trans_sept_1st_wire_or_ach_converted_df['Amount'] = trans_sept_1st_wire_or_ach_converted_df.apply(lambda row: row['Amount Paid'] / conversion_rates_df[row['Payment Currency']][row["Timestamp"].split(" ")[0]], axis=1)
trans_sept_1st_wire_or_ach_filtered = trans_sept_1st_wire_or_ach_converted_df[trans_sept_1st_wire_or_ach_converted_df['Amount'] < 1.0]
print("SIZE:", trans_sept_1st_wire_or_ach_filtered.shape[0])
q5_solucion = pd.DataFrame({"count": [trans_sept_1st_wire_or_ach_filtered.shape[0]]})
q5_solucion.to_csv("q5_solucion.csv", index=False)

SIZE: 9785
